# Spatial Cluster Preview For Original Events

This notebook loads original event locations, runs a selected spatial clustering method, and visualizes possible clusters before running HypoDD.

Reusable routines live in `cluster_preview.py`; this notebook is only the interactive control surface.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cluster_preview import (
    ClusteringConfig,
    cluster_summary,
    dbscan_eps_sweep,
    inventory_to_dataframe,
    load_events,
    plot_cluster_map,
    run_clustering,
    safe_output_name,
)

plt.rcParams["figure.figsize"] = (8, 7)
plt.rcParams["axes.grid"] = True

## Input Files And Clustering Parameters

Use `events.xml` if you already converted your CSV files to QuakeML. If not, set `EVENT_XML = None` and use `EVENT_CSV`.

In [ ]:
EVENT_XML = Path("events.xml")
STATION_XML = Path("stations.xml")
EVENT_CSV = Path("data/starting-event-Malyn.csv")

# Choose one of: "dbscan", "hdbscan", "agglomerative", "kmeans".
config = ClusteringConfig(
    method="dbscan",
    eps_km=3.0,
    min_samples=8,
    hdbscan_min_cluster_size=20,
    hdbscan_min_samples=8,
    agglomerative_distance_threshold_km=3.0,
    agglomerative_linkage="single",  # try "complete" to reduce bridge merging
    kmeans_n_clusters=4,
    kmeans_random_state=42,
)

EPS_VALUES_KM = [1.5, 2.0, 2.5, 3.0, 4.0, 5.0]

In [ ]:
events, event_source = load_events(event_xml=EVENT_XML, event_csv=EVENT_CSV)
stations = inventory_to_dataframe(STATION_XML)

print(f"Loaded {len(events)} events from {event_source}")
print(f"Loaded {len(stations)} stations from {STATION_XML if STATION_XML.exists() else 'none'}")

events.head()

## Selected Method Preview

In [ ]:
selected, selected_description = run_clustering(events, config)
display(cluster_summary(selected))

ax = plot_cluster_map(
    selected,
    stations=stations,
    title=f"Spatial clustering preview: {selected_description}",
)
plt.show()

## DBSCAN `eps_km` Sweep

This is useful for finding the distance threshold where two visually separate event clouds start to merge through bridge events.

In [ ]:
clustered_by_eps, eps_summary = dbscan_eps_sweep(
    events,
    EPS_VALUES_KM,
    min_samples=config.min_samples,
)
eps_summary

In [ ]:
ncols = 3
nrows = int(np.ceil(len(EPS_VALUES_KM) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows), squeeze=False)

for ax, eps_km in zip(axes.ravel(), EPS_VALUES_KM):
    clustered = clustered_by_eps[eps_km]
    row = eps_summary[eps_summary["eps_km"] == eps_km].iloc[0]
    plot_cluster_map(
        clustered,
        stations=stations,
        title=(
            f"eps={eps_km} km | clusters={int(row.clusters)} | "
            f"noise={int(row.noise_events)}"
        ),
        ax=ax,
    )
    ax.legend().remove()

for ax in axes.ravel()[len(EPS_VALUES_KM):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Save The Preview Labels

Cluster `-1` means noise/unclustered for methods that support noise labels.

In [ ]:
OUTPUT_CSV = Path(safe_output_name(selected_description))
selected.to_csv(OUTPUT_CSV, index=False)
OUTPUT_CSV